# Test Full Game Generation Workflow

This notebook calls the backend endpoint `POST /ai/game-generation`.

Flow covered:
1. Send `userId` (and optional `size`)
2. Backend generates game description with storytelling service
3. Backend generates HTML game and saves all data in MongoDB
4. Backend returns HTML to the client
5. Notebook renders returned HTML inline


In [1]:
import os
import json
import time
from typing import Any, Dict

import requests
from IPython.display import HTML, display

BASE_URL = os.getenv("BACKEND_BASE_URL", "http://localhost:3000")
GAME_ENDPOINT = f"{BASE_URL}/ai/game-generation"

print("Using endpoint:", GAME_ENDPOINT)


Using endpoint: http://localhost:3000/ai/game-generation


In [2]:
def generate_game_for_user(user_id: str, size: int = 640) -> Dict[str, Any]:
    payload = {
        "userId": user_id,
        "size": size,
    }

    started = time.time()
    response = requests.post(GAME_ENDPOINT, json=payload, timeout=600)
    elapsed = round(time.time() - started, 2)

    print(f"HTTP {response.status_code} in {elapsed}s")

    try:
        data = response.json()
    except Exception:
        print("Raw response:")
        print(response.text)
        raise

    if not response.ok:
        print(json.dumps(data, indent=2))
        response.raise_for_status()

    return data


In [4]:

USER_ID = "69b570a08e33a3fb8d0eb888"
SIZE = 640

if USER_ID == "PUT_REAL_USER_ID_HERE":
    raise ValueError("Set USER_ID to a valid user id before running this cell.")

result = generate_game_for_user(USER_ID, SIZE)

print("\nWorkflow response keys:", list(result.keys()))
print("Game ID:", result.get("gameId"))
print("Trace ID:", result.get("traceId"))
print("Attempts:", result.get("attempts"))
print("Created At:", result.get("createdAt"))


ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [ ]:
game_description = result.get("gameDescription", "")
game_html = result.get("html", "")

print("Description preview:\n")
print(game_description[:700] + ("..." if len(game_description) > 700 else ""))
print("\nHTML length:", len(game_html))


In [ ]:
# Render returned game HTML directly in notebook
display(HTML(game_html))


In [ ]:
# Optional: save generated HTML locally for inspection
output_file = "generated_game_preview.html"
with open(output_file, "w", encoding="utf-8") as f:
    f.write(game_html)

print("Saved:", output_file)
